In [41]:
from pynwb import NWBHDF5IO
import pynwb
from nlb_tools.nwb_interface import NWBDataset
import h5py
from collections import Counter
import numpy as np
import torch

In [ ]:
nwb_path = 'datasets/000128/sub-Jenkins/sub-Jenkins_ses-full_desc-train_behavior+ecephys.nwb'

def process(nwb_path, bin_size_ms = 5): 
    io = pynwb.NWBHDF5IO(nwb_path, 'r')
    nwbfile = io.read()
    
    units = nwbfile.units.to_dataframe()
    trials = nwbfile.trials.to_dataframe()
    
    num_neurons = len(units)
    dt = bin_size_ms / 1000.0
    processed_trials = []
    
    for i, trial in trials.iterrows():
        start = trial['start_time']
        stop = trial['stop_time']
        duration = start - stop
        
        time_bins = np.arange(start, stop, dt)
        num_bins = len(time_bins) - 1
        
        spikes = np.zeros((num_bins, num_neurons), dtype=np.float32)
        
        for neuron, spike_times in enumerate(units['spike_times']):
            trial_spikes = spike_times[(spike_times >= start) & (spike_times < stop)]
            counts, _ = np.histogram(trial_spikes, bins=time_bins)
            spikes[:, neuron] = counts
        
        trial_tensor = torch.from_numpy(spikes)
        processed_trials.append(trial_tensor)
        
    io.close()
    return processed_trials

t = process(nwb_path, bin_size_ms = 5)
print(t)


In [15]:
dataset = NWBDataset(filepath,"*train", split_heldout=False)

In [81]:
nwb_path = 'datasets/000128/sub-Jenkins/sub-Jenkins_ses-full_desc-train_behavior+ecephys.nwb'
nwb_path_test = 'datasets/000128/sub-Jenkins/sub-Jenkins_ses-full_desc-test_ecephys.nwb'

In [84]:
io = NWBHDF5IO(nwb_path, 'r')
io_test = NWBHDF5IO(nwb_path_test, 'r')
nwbfile_test = io_test.read()
nwbfile = io.read()

In [93]:
Counter(nwbfile.units.to_dataframe()['heldout'])

Counter({False: 137, True: 45})

In [113]:
len(nwbfile.trials)

2295

In [72]:
raw_velocity = nwbfile.processing['behavior']['hand_vel'].data[:]
vel_timestamps = nwbfile.processing['behavior']['hand_vel'].timestamps[:]


In [73]:
vel_timestamps

array([0.000000e+00, 1.000000e-03, 2.000000e-03, ..., 6.952298e+03,
       6.952299e+03, 6.952300e+03], shape=(6809920,))

In [32]:
trials = nwbfile.trials.to_dataframe()
trials

,start_time,stop_time,trial_type,trial_version,maze_id,success,target_on_time,go_cue_time,move_onset_time,rt,delay,num_targets,target_pos,num_barriers,barrier_pos,active_target,split
id,,,,,,,,,,,,,,,,,
0,0.0,3.321,25,2,84,True,0.880,1.478,1.905,427,598,3,"[[-111, -82], [-108, 81], [118, 72]]",8,"[[69, 31, 14, 99], [69, 54, 5, 101], [-62, -48...",2,val
1,3.4,6.521,3,1,3,True,4.291,4.739,5.280,541,448,1,"[[-116, -5]]",6,"[[-69, -16, 13, 69], [-120, -62, 83, 15], [95,...",0,val
2,6.6,9.856,22,1,66,True,7.471,7.969,8.346,377,498,1,"[[-82, -86]]",9,"[[34, -41, 86, 8], [9, -42, 33, 19], [7, -41, ...",0,train
3,9.9,12.946,29,2,100,True,10.853,11.335,11.752,417,482,3,"[[-109, 2], [2, 82], [132, -65]]",9,"[[-9, 52, 43, 8], [-50, 91, 14, 64], [-133, -5...",1,train
4,13.0,15.481,21,0,65,True,13.687,14.235,14.507,272,548,1,"[[27, 82]]",0,[],0,val
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2290,6936.6,6939.796,34,1,91,True,6937.362,6938.277,6938.585,308,915,1,"[[116, -77]]",7,"[[66, -43, 30, 9], [-66, 1, 11, 70], [-35, 50,...",0,train
2291,6939.9,6942.736,15,1,75,True,6940.717,6941.265,6941.641,376,548,1,"[[133, -81]]",9,"[[-33, 47, 37, 6], [-77, 48, 61, 11], [-64, -2...",0,train
2292,6942.8,6945.766,23,0,67,True,6943.465,6944.396,6944.714,318,931,1,"[[94, -86]]",0,[],0,train


In [34]:
dataset.data[~dataset.data[('cursor_pos','x')].isna()]

/hpc/home/zy231/.local/lib/python3.14/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


signal_type             cursor_pos            eye_pos           hand_pos  \
channel                          x          y       x      y           x   
clock_time                                                                 
0 days 00:00:00          -0.900000  -5.700000     7.2    2.0   -0.714908   
0 days 00:00:00.001000   -0.907457  -5.687027     7.2    2.1   -0.717532   
0 days 00:00:00.002000   -0.912768  -5.672115     7.6    1.2   -0.720323   
0 days 00:00:00.003000   -0.914050  -5.653433     7.4    1.4   -0.723278   
0 days 00:00:00.004000   -0.909980  -5.629617     7.4    3.6   -0.726362   
...                            ...        ...     ...    ...         ...   
0 days 01:55:52.296000 -114.378901 -79.712313   -95.0 -117.5 -114.334012   
0 days 01:55:52.297000 -114.366164 -79.728485   -94.9 -117.4 -114.333252   
0 days 01:55:52.298000 -114.365911 -79.749577   -94.6 -117.7 -114.332816   
0 days 01:55:52.299000 -114.378419 -79.774473   -94.8 -117.7 -114.332814   
0 days 01:55:52.300000 -114.400000 -79.800000   -97.8 -118.2 -114.333242   

signal_type                         hand_vel            spikes       ...       \
channel                          y         x          y   1011 1021  ... 2861   
clock_time                                                           ...        
0 days 00:00:00         -40.526123 -2.624567  29.977111    0.0  0.0  ...  0.0   
0 days 00:00:00.001000  -40.496146 -2.707321  30.577662    0.0  0.0  ...  0.0   
0 days 00:00:00.002000  -40.464968 -2.872729  31.744164    0.0  0.0  ...  0.0   
0 days 00:00:00.003000  -40.432658 -3.019660  32.847931    0.0  0.0  ...  0.0   
0 days 00:00:00.004000  -40.399272 -3.059403  33.895227    0.0  0.0  ...  0.0   
...                            ...       ...        ...    ...  ...  ...  ...   
0 days 01:55:52.296000 -114.809976  0.905895  -0.883716    0.0  0.0  ...  0.0   
0 days 01:55:52.297000 -114.810622  0.598148  -0.420075    0.0  0.0  ...  0.0   
0 days 01:55:52.298000 -114.810816  0.218816   0.012961    0.0  0.0  ...  0.0   
0 days 01:55:52.299000 -114.810596 -0.212940   0.393580    0.0  0.0  ...  0.0   
0 days 01:55:52.300000 -114.810029 -0.427820   0.566803    0.0  0.0  ...  0.0   

signal_type                                                          
channel                2862 2871 2881 2882 2911 2931 2941 2951 2961  
clock_time                                                           
0 days 00:00:00         0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
0 days 00:00:00.001000  0.0  0.0  0.0  0.0  0.0  1.0  0.0  0.0  0.0  
0 days 00:00:00.002000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
0 days 00:00:00.003000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
0 days 00:00:00.004000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
...                     ...  ...  ...  ...  ...  ...  ...  ...  ...  
0 days 01:55:52.296000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
0 days 01:55:52.297000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
0 days 01:55:52.298000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
0 days 01:55:52.299000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
0 days 01:55:52.300000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  

[6809920 rows x 190 columns]

In [30]:
dataset.data

/hpc/home/zy231/.local/lib/python3.14/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


signal_type            cursor_pos           eye_pos       hand_pos             \
channel                         x         y       x    y         x          y   
clock_time                                                                      
0 days 00:00:00         -0.900000 -5.700000     7.2  2.0 -0.714908 -40.526123   
0 days 00:00:00.001000  -0.907457 -5.687027     7.2  2.1 -0.717532 -40.496146   
0 days 00:00:00.002000  -0.912768 -5.672115     7.6  1.2 -0.720323 -40.464968   
0 days 00:00:00.003000  -0.914050 -5.653433     7.4  1.4 -0.723278 -40.432658   
0 days 00:00:00.004000  -0.909980 -5.629617     7.4  3.6 -0.726362 -40.399272   
...                           ...       ...     ...  ...       ...        ...   
0 days 01:55:49.295000        NaN       NaN     NaN  NaN       NaN        NaN   
0 days 01:55:49.296000        NaN       NaN     NaN  NaN       NaN        NaN   
0 days 01:55:49.297000        NaN       NaN     NaN  NaN       NaN        NaN   
0 days 01:55:49.298000        NaN       NaN     NaN  NaN       NaN        NaN   
0 days 01:55:49.299000        NaN       NaN     NaN  NaN       NaN        NaN   

signal_type             hand_vel            spikes       ...                 \
channel                        x          y   1011 1021  ... 2861 2862 2871   
clock_time                                               ...                  
0 days 00:00:00        -2.624567  29.977111    0.0  0.0  ...  0.0  0.0  0.0   
0 days 00:00:00.001000 -2.707321  30.577662    0.0  0.0  ...  0.0  0.0  0.0   
0 days 00:00:00.002000 -2.872729  31.744164    0.0  0.0  ...  0.0  0.0  0.0   
0 days 00:00:00.003000 -3.019660  32.847931    0.0  0.0  ...  0.0  0.0  0.0   
0 days 00:00:00.004000 -3.059403  33.895227    0.0  0.0  ...  0.0  0.0  0.0   
...                          ...        ...    ...  ...  ...  ...  ...  ...   
0 days 01:55:49.295000       NaN        NaN    NaN  NaN  ...  NaN  NaN  NaN   
0 days 01:55:49.296000       NaN        NaN    NaN  NaN  ...  NaN  NaN  NaN   
0 days 01:55:49.297000       NaN        NaN    NaN  NaN  ...  NaN  NaN  NaN   
0 days 01:55:49.298000       NaN        NaN    NaN  NaN  ...  NaN  NaN  NaN   
0 days 01:55:49.299000       NaN        NaN    NaN  NaN  ...  NaN  NaN  NaN   

signal_type                                                
channel                2881 2882 2911 2931 2941 2951 2961  
clock_time                                                 
0 days 00:00:00         0.0  0.0  0.0  0.0  0.0  0.0  0.0  
0 days 00:00:00.001000  0.0  0.0  0.0  1.0  0.0  0.0  0.0  
0 days 00:00:00.002000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
0 days 00:00:00.003000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
0 days 00:00:00.004000  0.0  0.0  0.0  0.0  0.0  0.0  0.0  
...                     ...  ...  ...  ...  ...  ...  ...  
0 days 01:55:49.295000  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
0 days 01:55:49.296000  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
0 days 01:55:49.297000  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
0 days 01:55:49.298000  NaN  NaN  NaN  NaN  NaN  NaN  NaN  
0 days 01:55:49.299000  NaN  NaN  NaN  NaN  NaN  NaN  NaN  

[6952301 rows x 190 columns]

In [45]:
sum(dataset.data.spikes.iloc[6952300])

nan

In [46]:
from collections import Counter

In [47]:
Counter(dataset.trial_info['split'])

Counter({'train': 1721, 'val': 574})

In [11]:
dataset.trial_info

,trial_id,start_time,end_time,trial_type,trial_version,maze_id,success,target_on_time,go_cue_time,move_onset_time,rt,delay,num_targets,target_pos,num_barriers,barrier_pos,active_target,split
0,0,0 days 00:00:00,0 days 00:00:03.321000,25,2,84,True,0 days 00:00:00.880000,0 days 00:00:01.478000,0 days 00:00:01.905000,427,598,3,"[[-111, -82], [-108, 81], [118, 72]]",8,"[[69, 31, 14, 99], [69, 54, 5, 101], [-62, -48...",2,val
1,1,0 days 00:00:03.400000,0 days 00:00:06.521000,3,1,3,True,0 days 00:00:04.291000,0 days 00:00:04.739000,0 days 00:00:05.280000,541,448,1,"[[-116, -5]]",6,"[[-69, -16, 13, 69], [-120, -62, 83, 15], [95,...",0,val
2,2,0 days 00:00:06.600000,0 days 00:00:09.856000,22,1,66,True,0 days 00:00:07.471000,0 days 00:00:07.969000,0 days 00:00:08.346000,377,498,1,"[[-82, -86]]",9,"[[34, -41, 86, 8], [9, -42, 33, 19], [7, -41, ...",0,train
3,3,0 days 00:00:09.900000,0 days 00:00:12.946000,29,2,100,True,0 days 00:00:10.853000,0 days 00:00:11.335000,0 days 00:00:11.752000,417,482,3,"[[-109, 2], [2, 82], [132, -65]]",9,"[[-9, 52, 43, 8], [-50, 91, 14, 64], [-133, -5...",1,train
4,4,0 days 00:00:13,0 days 00:00:15.481000,21,0,65,True,0 days 00:00:13.687000,0 days 00:00:14.235000,0 days 00:00:14.507000,272,548,1,"[[27, 82]]",0,[],0,val
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2290,2290,0 days 01:55:36.600000,0 days 01:55:39.796000,34,1,91,True,0 days 01:55:37.362000,0 days 01:55:38.277000,0 days 01:55:38.585000,308,915,1,"[[116, -77]]",7,"[[66, -43, 30, 9], [-66, 1, 11, 70], [-35, 50,...",0,train
2291,2291,0 days 01:55:39.900000,0 days 01:55:42.736000,15,1,75,True,0 days 01:55:40.717000,0 days 01:55:41.265000,0 days 01:55:41.641000,376,548,1,"[[133, -81]]",9,"[[-33, 47, 37, 6], [-77, 48, 61, 11], [-64, -2...",0,train
2292,2292,0 days 01:55:42.800000,0 days 01:55:45.766000,23,0,67,True,0 days 01:55:43.465000,0 days 01:55:44.396000,0 days 01:55:44.714000,318,931,1,"[[94, -86]]",0,[],0,train
2293,2293,0 days 01:55:45.800000,0 days 01:55:49.201000,25,2,84,True,0 days 01:55:46.631000,0 days 01:55:46.663000,0 days 01:55:47.616000,953,32,3,"[[-111, -82], [-108, 81], [118, 72]]",8,"[[69, 31, 14, 99], [69, 54, 5, 101], [-62, -48...",2,val


In [10]:
train_data_path = 'datasets/000128/sub-Jenkins/sub-Jenkins_ses-full_desc-train_behavior+ecephys.nwb'

In [13]:
with h5py.File(train_data_path, "r") as f:
    print(f.keys())


<KeysViewHDF5 ['acquisition', 'analysis', 'file_create_date', 'general', 'identifier', 'intervals', 'processing', 'session_description', 'session_start_time', 'specifications', 'stimulus', 'timestamps_reference_time', 'units']>
